In [2]:
from sklearn.decomposition import TruncatedSVD
from sklearn.datasets import fetch_20newsgroups_vectorized

# 加载文本数据 (TF-IDF 矩阵)
data = fetch_20newsgroups_vectorized(subset='train')
X = data.data  # 稀疏矩阵，形状约为 (11314, 130107)

# 使用 TruncatedSVD 降维到 100 维 (相当于 LSA)
# 注意：TruncatedSVD 不需要数据中心化，直接处理稀疏矩阵
svd = TruncatedSVD(n_components=100, random_state=42)
X_reduced = svd.fit_transform(X)

print(f"降维前维度: {X.shape}")
print(f"降维后维度: {X_reduced.shape}")
print(f"保留的方差比例: {svd.explained_variance_ratio_.sum():.4f}")

降维前维度: (11314, 130107)
降维后维度: (11314, 100)
保留的方差比例: 0.3999


In [3]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import svds

# 1. 模拟用户-物品评分矩阵 (5个用户, 6个物品, 0表示未评分)
R = np.array([
    [5, 3, 0, 1, 0, 0],
    [4, 0, 0, 1, 0, 0],
    [1, 1, 0, 5, 0, 0],
    [0, 0, 5, 0, 4, 5],
    [0, 0, 4, 0, 5, 4]
], dtype=float)

# 2. 数据预处理：减去用户均值 (中心化)
user_means = np.mean(R, axis=1, where=R>0)
R_centered = R - user_means[:, np.newaxis]
R_centered[R == 0] = 0 # 未评分的保持为0

# 3. 转换为稀疏矩阵并进行截断 SVD (k=2)
R_sparse = sp.csr_matrix(R_centered)
k = 2
U, sigma, Vt = svds(R_sparse, k=k)
sigma = np.diag(sigma)

# 4. 重构评分矩阵
R_pred_centered = np.dot(np.dot(U, sigma), Vt)
R_pred = R_pred_centered + user_means[:, np.newaxis]

print("原始评分矩阵:\n", R)
print("\nSVD预测的评分矩阵 (包含未评分的预测值):\n", np.round(R_pred, 2))
# 观察 R_pred 中原本为 0 的位置，现在有了预测评分！

原始评分矩阵:
 [[5. 3. 0. 1. 0. 0.]
 [4. 0. 0. 1. 0. 0.]
 [1. 1. 0. 5. 0. 0.]
 [0. 0. 5. 0. 4. 5.]
 [0. 0. 4. 0. 5. 4.]]

SVD预测的评分矩阵 (包含未评分的预测值):
 [[5.   3.   3.   1.   3.   3.  ]
 [4.   2.5  2.5  1.   2.5  2.5 ]
 [1.   1.   2.33 5.   2.33 2.33]
 [4.67 4.67 4.67 4.67 4.67 4.67]
 [4.33 4.33 4.33 4.33 4.33 4.33]]


In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 加载数据
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 初始化并训练 (既可以作为分类器，也可以作为转换器)
lda = LinearDiscriminantAnalysis(n_components=2) 
lda.fit(X_train, y_train)

# 作为分类器直接预测
y_pred = lda.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# 作为降维工具转换数据
X_transformed = lda.transform(X_train) # 形状变为 (n_samples, 2)

Accuracy: 1.0000


In [1]:
import numpy as np

A = np.array([[2, 1],
              [5, 3]])

# ✅ 推荐：np.linalg.inv
A_inv = np.linalg.inv(A)
print(A_inv)
# [[ 3. -1.]
#  [-5.  2.]]

# 验证
identity_check = A @ A_inv
print(np.allclose(identity_check, np.eye(2)))  # True ✅

# ⚠️ 奇异矩阵会报错
B = np.array([[1, 2],
              [2, 4]])       # det=0, 不可逆
try:
    np.linalg.inv(B)
except np.linalg.LinAlgError as e:
    print(f"Singular matrix: {e}")

# 💡 实际求解 Ax=b 时，不要用 inv！用 solve 更稳定更快
b = np.array([3, 8])
x = np.linalg.solve(A, b)   # ✅ 推荐
x_bad = np.linalg.inv(A) @ b # ❌ 数值不稳定，速度慢

[[ 3. -1.]
 [-5.  2.]]
True
Singular matrix: Singular matrix
